In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/dataset.csv")


In [2]:
df = df.drop(columns=["Unnamed: 0", "timestamp"])


In [3]:
df["date"] = pd.to_datetime(df["date"])


In [4]:
df = df.sort_values(["crypto_name", "date"]).reset_index(drop=True)


In [5]:
df["volume"] = df["volume"].replace(0, np.nan)
df["volume"] = df.groupby("crypto_name")["volume"].fillna(method="ffill")


C:\Users\bhoom\AppData\Local\Temp\ipykernel_23232\1305838446.py:2: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  df["volume"] = df.groupby("crypto_name")["volume"].fillna(method="ffill")
C:\Users\bhoom\AppData\Local\Temp\ipykernel_23232\1305838446.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["volume"] = df.groupby("crypto_name")["volume"].fillna(method="ffill")


In [6]:

df.isnull().sum()


open             0
high             0
low              0
close            0
volume         633
marketCap        0
crypto_name      0
date             0
dtype: int64

In [7]:
df = df.dropna()


In [8]:
df = df.sort_values(["crypto_name", "date"]).reset_index(drop=True)


In [9]:
df = df.drop_duplicates()

price_cols = ["open", "high", "low", "close"]
df = df[(df[price_cols] > 0).all(axis=1)]


In [10]:
import numpy as np

df["log_return"] = (
    df.groupby("crypto_name")["close"]
      .apply(lambda x: np.log(x / x.shift(1)))
      .reset_index(level=0, drop=True)
)


In [11]:
df["volatility_7d"] = (
    df.groupby("crypto_name")["log_return"]
      .apply(lambda x: x.rolling(7).std())
      .reset_index(level=0, drop=True)
)


In [12]:
df["close_lag_1"] = df.groupby("crypto_name")["close"].shift(1)
df["close_lag_7"] = df.groupby("crypto_name")["close"].shift(7)

df["vol_lag_1"] = df.groupby("crypto_name")["volatility_7d"].shift(1)


In [13]:
df["volatility_14d"] = (
    df.groupby("crypto_name")["log_return"]
    .rolling(14)
    .std()
    .reset_index(level=0, drop=True)
)

df["ma_7"] = (
    df.groupby("crypto_name")["close"]
    .rolling(7)
    .mean()
    .reset_index(level=0, drop=True)
)


In [14]:
df["tr"] = np.maximum(
    df["high"] - df["low"],
    np.maximum(
        abs(df["high"] - df["close"].shift(1)),
        abs(df["low"] - df["close"].shift(1))
    )
)

df["atr_14"] = (
    df.groupby("crypto_name")["tr"]
      .transform(lambda x: x.rolling(14).mean())
)


In [15]:
df["bb_middle"] = (
    df.groupby("crypto_name")["close"]
      .transform(lambda x: x.rolling(20).mean())
)

df["bb_std"] = (
    df.groupby("crypto_name")["close"]
      .transform(lambda x: x.rolling(20).std())
)

df["bb_upper"] = df["bb_middle"] + 2 * df["bb_std"]
df["bb_lower"] = df["bb_middle"] - 2 * df["bb_std"]


In [16]:
df["hl_range"] = (df["high"] - df["low"]) / df["close"]
df["oc_change"] = (df["close"] - df["open"]) / df["open"]
df["liquidity_ratio"] = df["volume"] / df["marketCap"]


In [17]:
df = df.dropna().reset_index(drop=True)


In [18]:
df = df.drop_duplicates()

price_cols = ["open", "high", "low", "close"]
df = df[(df[price_cols] > 0).all(axis=1)]


In [19]:
df = pd.get_dummies(df, columns=["crypto_name"], drop_first=True)


In [25]:
final_columns = [
    "open", "high", "low", "close",
    "volume", "marketCap",

    "close_lag_1", "close_lag_7",
    "vol_lag_1", "volatility_14d",
    "ma_7",

    "bb_middle", "bb_upper", "bb_lower",   # ✅ Bollinger Bands

    "hl_range", "oc_change", "liquidity_ratio",

    "volatility_7d"   # 🎯 TARGET
]

df_final = df[final_columns]


In [26]:
import os

os.makedirs("../data/processed", exist_ok=True)


In [27]:
df_final.to_csv("../data/processed/crypto_volatility_processed.csv", index=False)


In [28]:
pd.read_csv("../data/processed/crypto_volatility_processed.csv").head()


,open,high,low,close,volume,marketCap,close_lag_1,close_lag_7,vol_lag_1,volatility_14d,ma_7,bb_middle,bb_upper,bb_lower,hl_range,oc_change,liquidity_ratio,volatility_7d
0,39.455022,40.928509,37.601201,37.904761,4.397045e+07,4.048626e+08,39.455022,40.746972,0.094234,0.083418,37.204395,42.630075,54.211843,31.048308,0.087781,-0.039292,0.108606,0.095072
1,37.904763,39.163864,35.071968,36.033922,5.030722e+07,3.848801e+08,37.904761,36.012317,0.095072,0.081252,37.207481,42.311691,54.264071,30.359312,0.113557,-0.049356,0.130709,0.083948
2,36.033914,38.869182,34.469635,34.871531,5.858131e+07,3.724646e+08,36.033922,32.180017,0.083948,0.081255,37.591983,42.051069,54.427760,29.674378,0.126164,-0.032258,0.157280,0.070447
3,34.871536,35.564022,32.000059,32.394083,6.053387e+07,3.460028e+08,34.871531,35.160357,0.070447,0.081611,37.196801,41.482550,54.553001,28.412100,0.110019,-0.071045,0.174952,0.067485
4,32.394082,33.155484,28.240773,29.098334,6.630479e+07,3.108008e+08,32.394083,38.031651,0.067485,0.077185,35.920613,40.596580,54.518768,26.674391,0.168900,-0.101739,0.213335,0.062442
